In [1]:
# imports
import datetime
import time

import matplotlib.pyplot as plt
import numpy as np
import pyodbc
import pandas as pd
from pandas import DataFrame
from pyodbc import Row

In [2]:
def time_to_seconds(_time: datetime.time) -> int:
    return _time.hour * 3600 + _time.minute * 60 + _time.second


def format_time_to_ms(_time: float) -> str:
    return f"{_time * 1000:.2f}ms"

In [3]:
# configs
name_map: dict[str, str] = {
    "dow": "day_of_week",
    "dom": "day_of_month",
    "doy": "day_of_year",
    "wn": "week_number",
    "mn": "month_number",
    "qn": "quarter_number",
    "yn": "year_number",
}


def sql_grain_pt(grain: str) -> str:
    return f"""
        WITH {grain}_purchase_times AS (
            SELECT
                o.{grain},
                CONVERT(TIME, o.order_purchase_timestamp) AS purchase_time,
                DATEDIFF(SECOND, o.order_purchase_timestamp, order_approved_at)
                    AS diff_purchase_to_approve_s
            FROM sales.vw_orders_practical AS o
        )
        SELECT
            o.*,
            COUNT(*) AS order_count
        FROM {grain}_purchase_times AS o
        GROUP BY
            o.{grain},
            o.purchase_time,
            o.diff_purchase_to_approve_s
        ORDER BY
            o.{grain},
            o.purchase_time,
            o.diff_purchase_to_approve_s;
        """


# connect
driver = "ODBC Driver 17 for SQL Server"
server = "localhost"
database = "olist"

connstring: str = f"""
    DRIVER={{{driver}}};
    SERVER={server};
    DATABASE={database};
    Trusted_Connection=yes;
"""
tik_connect = time.perf_counter()
conn = pyodbc.connect(connstring, timeout=60)
cursor = conn.cursor()
tok_connect = time.perf_counter()

# get day table
day_pt_rows: list[Row] = cursor.execute("""
    SELECT
        CONVERT(TIME, o.order_purchase_timestamp) AS purchase_time,
        COUNT(*) AS order_count
    FROM sales.vw_orders_practical AS o
    GROUP BY CONVERT(TIME, o.order_purchase_timestamp)
    ORDER BY purchase_time;
""").fetchall()

day_pt_columns: list[str] = [description[0] for description in cursor.description]
df_pt_day: DataFrame = pd.DataFrame(
    (tuple(r) for r in day_pt_rows), columns=day_pt_columns
)

# get other tables
dfs_pt: dict[str, DataFrame] = {}

for name in name_map:
    df_name = f"{name}"
    grain = name_map[name]

    sql = sql_grain_pt(grain=grain)
    grain_pt_rows: list[Row] = cursor.execute(sql).fetchall()

    grain_pt_columns: list[str] = [description[0] for description in cursor.description]
    df_grain_pt = pd.DataFrame(
        (tuple(r) for r in grain_pt_rows), columns=grain_pt_columns
    )

    dfs_pt[df_name] = df_grain_pt

conn.close()

print(f"Connect to DB: {format_time_to_ms(tok_connect - tik_connect)}")

Connect to DB: 76.65ms


In [4]:
# clean
time_to_seconds_column_name = "purchase_time_s"

tik_day = time.perf_counter()
if not df_pt_day.columns.str.contains(time_to_seconds_column_name).any():
    df_pt_day[time_to_seconds_column_name] = df_pt_day["purchase_time"].apply(
        time_to_seconds
    )
tok_day = time.perf_counter()

tik_other = time.perf_counter()
for i in dfs_pt:
    if not dfs_pt[i].columns.str.contains(time_to_seconds_column_name).any():
        dfs_pt[i][time_to_seconds_column_name] = dfs_pt[i]["purchase_time"].apply(
            time_to_seconds
        )
tok_other = time.perf_counter()

print(f"Clean `df_pt_day`: {format_time_to_ms(tok_day - tik_day)}")
print(f"Clean `dfs_pt`: {format_time_to_ms(tok_other - tik_other)}")

Clean `df_pt_day`: 24.05ms
Clean `dfs_pt`: 252.35ms
